# POP77001 Assignment 1: R

options(jupyter.rich_display = FALSE)

## Before Submission

- Do not use external libraries unless explicitly specified.
- Make sure that you can run all cells without errors.
- You can do it by clicking `Kernel`, `Restart & Run All` in the menu
  above.
- Make sure that you save the output by pressing Command+S / CTRL+S.
- Rename the file from `01_assignment.ipynb` to
  `01_assignment_lastname_firstname_studentnumber.ipynb`
- Fill in and upload Assignment Cover Sheet together with your
  submission.

## Exercise 1: Attributes and Factors

Suppose you read in a variable which represents education of survey
respondents. After consulting the codebook for this survey you
determined that specific values that the variable takes are coded as
follows:

- `0` - No formal education
- `1` - Primary Education
- `2` - Secondary Education
- `3` - College Education
- `4` - Post-graduate Education

Convert the variable `educ` below into a factor variable with each value
coded explicitly as one of the education levels. Tabulate the variable
using `table()` function. Change the order of the levels of this factor
variable such that the most common education level is the first
(baseline/reference) category (you can leave other education levels in
their original order). Tabulate the variable again.

In [1]:
set.seed(2026)
educ <- rpois(1000, 2)
educ <- ifelse(educ > 4, 4, educ)

In [34]:
# Exercise 1:

set.seed(2026)
educ <- rpois(1000, 2)
educ <- ifelse(educ > 4, 4, educ)


# Your code goes here
#Right now educ is just numbers (0, 1, 2, 3, 4). it doesn't know those numbers are categories with names. 
#The task is to:
	#1. Tell R that 0 means "No formal education", 1 means "Primary", and so on.
	#2. Count how many people fall into each category.
	#3. Reorder the categories so the most common one comes first.
	#4. Count again to see the change.

# Step 1: turn the numeric variable into a factor.
# 'levels' = the values that actually appear in the data (0 to 4)
# 'labels' = the names we want R to use for each of those values, in the same order
educ_f <- factor(educ,
                 levels = c(0, 1, 2, 3, 4),
                 labels = c("No formal education",
                            "Primary",
                            "Secondary",
                            "College",
                            "Post-graduate"))


# Step 2: count how many respondents are in each category
table(educ_f)
#print numbers here:

#step 3:
# table() gives counts, which.max() finds the position of the biggest count,
# and names() pulls out the label at that position.
most_common <- names(which.max(table(educ_f)))
most_common   # check it prints "Secondary"

#Step 4:
# Move that category to the front (making it the baseline/reference level)
educ_f <- relevel(educ_f, ref = most_common)

# Tabulate again
table(educ_f)




## Exercise 2: Control Flow

[Skewness](https://en.wikipedia.org/wiki/Skewness) is an important
concept in data analysis. Many methods of statistical analysis assume
that the variables roughly follow a [normal
distribution](https://en.wikipedia.org/wiki/Normal_distribution).

In this exercise you will need to write R code to determine whether each
of the variables is approximately normally distributed and if not,
report the direction of skew (left/negative or right/positive). The
easiest way to do so is by comparing means and medians of the
distribution for each variable.

Look at the code below. Each variable is stored as a vector in a list.
Use loop(s) and conditional statements to analyse them.

set.seed(2026)
vars <- list(
  var1 = rexp(1000),
  var2 = rnorm(1000, -2.75, 3),
  var3 = rchisq(1000, 2),
  var4 = rt(1000, 30),
  var5 = rbeta(1000, 5, 2)
)

In [11]:
# Exercise 2:

set.seed(2026)
vars <- list(
var1 = rexp(1000),
var2 = rnorm(1000, -2.75, 3),
var3 = rchisq(1000, 2),
var4 = rt(1000, 30),
var5 = rbeta(1000, 5, 2)
)


# Your code goes here
#For each of the five variables, decide whether it looks roughly normal (symmetric, bell-shaped), or If it's skewed and what direction the skew is 

	#Mean ≈ median: roughly symmetric, so approximately normal
	#Mean > median: a long tail to the right, so right (positive) skew
	#Mean < median: a long tail to the left, so left (negative) skew


#Step 1: Look at the data
# Pull out one variable and compute its mean and median
mean(vars$var1)
median(vars$var1)


#Step 2: compare the median and mean of Vars list
#Var1 is right-skewed: its mean (1.041) is well above its median (0.731). var3 shows the same pattern (2.023 vs 1.409), so it's right-skewed

#Step 3: how big is the gaps ?? 
#use a fixed cutoff like "differ by less than 0.05", var5 would pass as normal and var2 would fail as var2's mean and median differ by about 0.116, where as for var5 its only 0.024
# but the variables are on different scales. 

sd(vars$var2)          
sd(vars$var5)

#var2 has a standard deviation of about 3, so 0.116 is tiny in the grand scheme of things.
#var5 var5 is between 0 and 1 with a standard deviation of about 0.16, so thats a lot larger 

# To fix this, divide the gap by the standard deviation:
# skew_score <- (mean(x) - median(x)) / sd(x)
# This says how many standard deviations apart the mean and median are.


#step 4: conditionals 

#for this step- i choose to use if the absolute value is below about 0.1, treat it as roughly symmetric, otherwise its skewed


for (v in names(vars)) {
  x <- vars[[v]]
  skew_score <- (mean(x) - median(x)) / sd(x)

  if (abs(skew_score) < 0.1) {
    verdict <- "approximately normal"
  } else if (skew_score > 0) {
    verdict <- "right (positive) skew"
  } else {
    verdict <- "left (negative) skew"
  }

  cat(v, "| score:", round(skew_score, 3), "|", verdict, "\n")
}



[1] 1.040663

[1] 0.7311286

[1] 3.002456

[1] 0.1588153

var1 | score: 0.3 | right (positive) skew 
var2 | score: -0.039 | approximately normal 
var3 | score: 0.315 | right (positive) skew 
var4 | score: 0.009 | approximately normal 
var5 | score: -0.148 | left (negative) skew 


## Exercise 3: Formalising Expectations

[Student’s t-test](https://en.wikipedia.org/wiki/Student's_t-test)
(developed by [William
Gosset](https://en.wikipedia.org/wiki/William_Sealy_Gosset) when working
at the Guinness Brewery) is a classical tool for testing differences in
means between groups when samples are small. It is a very simple test to
implement and understand, but it is also very powerful and is used in
many applications.

Re-visit the function for calculating t-test from the tutorial. How can
a call to this function produce invalid output? What are your
expectations about the input? Re-write the function below to check
user’s input and handle situations when a user provides incorrect
values.

In [2]:
# Exercise 3:
t_test <- function(x, y) {
  # Calculate the means of the two samples
  mean_x <- mean(x)
  mean_y <- mean(y)
  
  # Calculate the variances of the two samples
  var_x <- var(x)
  var_y <- var(y)
  
  # Calculate the sample sizes
  n_x <- length(x)
  n_y <- length(y)
  
  # Calculate the sample standard errors
  se_x <- sqrt(var_x/n_x)
  se_y <- sqrt(var_y/n_y)
  se <- sqrt(se_x^2 + se_y^2)
  
  # Calculate the t-statistic
  t_stat <- (mean_x - mean_y) / se
  
  # Calculate the degrees of freedom
  df <- se^4/(se_x^4/(n_x - 1) + se_y^4/(n_y - 1))
  
  # Calculate the p-value
  p_val <- 2 * (1 - pt(abs(t_stat), df))
  
  # Create a result list
  res <- list(
    t_statistic = t_stat,
    degrees_of_freedom = df,
    p_value = p_val
  )
  
  return(res)
}
# Your code goes here
    #The task is to:
    	#1. Find ways a user could call t_test() that produce bad answers
    	#2. Write down what the function expects from its input 
                #(the "expectations").
        #3. Add checks at the top that stop the function with a clear message when an expectation is broken.

#Step 1: Run the definition

t_test <- function(x, y) {
  mean_x <- mean(x)
  mean_y <- mean(y)
  var_x <- var(x)
  var_y <- var(y)
  n_x <- length(x)
  n_y <- length(y)
  se_x <- sqrt(var_x/n_x)
  se_y <- sqrt(var_y/n_y)
  se <- sqrt(se_x^2 + se_y^2)
  t_stat <- (mean_x - mean_y) / se
  df <- se^4/(se_x^4/(n_x - 1) + se_y^4/(n_y - 1))
  p_val <- 2 * (1 - pt(abs(t_stat), df))
  res <- list(
    t_statistic = t_stat,
    degrees_of_freedom = df,
    p_value = p_val
  )
  return(res)
}





## now lets run some bad inputs 
t_test(c(1, 2, 3, 4, 5), c(2, 4, 6, 8, 10))   # should still work

t_test(c("apple", "bananas", "cherries"), c(1, 2, 3)) 
t_test(c(NA, 2, NA, 4), c(5, 1, 7, 1))
t_test(5, c(1, 2, 3))
t_test(numeric(0), c(1, 0, 3))
t_test(c(3, 3, 3), c(3, 3, 3))


# Without checks, bad input (text, NA values, fewer than 2 observations, or two constant samples) returns NA/NaN instead of an error.
# Expectations: x and y are numeric, contain no NA/NaN/Inf, each has at least 2 observations, and at least one of them has variation.

##Step 3: make some ground rules 

t_test <- function(x, y) {

  # Expectation 1: both inputs must be numeric
  if (!is.numeric(x) || !is.numeric(y)) {
    break("Both x and y must be numeric vectors.")
      ### make sure no apples, bananas, or cherries are in the t_test
  }

  # Expectation 2: no NA, NaN or Inf values
  # is.finite() is FALSE for all of those, so one check covers them
  if (any(!is.finite(x)) || any(!is.finite(y))) {
    break("x and y must not contain missing or infinite values.")
      ###no Null or NAs here 
  }

  # Expectation 3: at least 2 observations in each sample
  if (length(x) < 2 || length(y) < 2) {
    break("Each sample needs at least 2 observations.")
  }

  # Expectation 4: at least one sample must vary, or the standard error is 0
  if (var(x) == 0 && var(y) == 0) {
    break("Both samples are constant, so the t-test is undefined.")
  }

  # ---- the original calculation, unchanged ----
  mean_x <- mean(x)
  mean_y <- mean(y)
  var_x <- var(x)
  var_y <- var(y)
  n_x <- length(x)
  n_y <- length(y)
  se_x <- sqrt(var_x/n_x)
  se_y <- sqrt(var_y/n_y)
  se <- sqrt(se_x^2 + se_y^2)
  t_stat <- (mean_x - mean_y) / se
  df <- se^4/(se_x^4/(n_x - 1) + se_y^4/(n_y - 1))
  p_val <- 2 * (1 - pt(abs(t_stat), df))

  res <- list(
    t_statistic = t_stat,
    degrees_of_freedom = df,
    p_value = p_val
  )
  return(res)
}

t_test(c(1, 2, 3, 4, 5), c(2, 4, 6, 8, 10))   # should still work

t_test(c("a", "b", "c"), c(1, 2, 3))
try(t_test(c(1, 2, NA, 4), c(5, 6, 7, 8)))
try(t_test(5, c(1, 2, 3)))
try(t_test(numeric(0), c(1, 2, 3)))
try(t_test(c(3, 3, 3), c(3, 3, 3)))


#Step 3
# Only ONE sample is constant: this should still work, since se > 0
t_test(c(3, 3, 3), c(1, 2, 3))

# Compare against R's built-in t-test
t.test(c(1, 2, 3, 4, 5), c(2, 4, 6, 8, 10))


$t_statistic
[1] -1.897367

$degrees_of_freedom
[1] 5.882353

$p_value
[1] 0.1075312

Warning message in mean.default(x):
“argument is not numeric or logical: returning NA”
Warning message in var(x):
“NAs introduced by coercion”


$t_statistic
[1] NA

$degrees_of_freedom
[1] NA

$p_value
[1] NA

$t_statistic
[1] NA

$degrees_of_freedom
[1] NA

$p_value
[1] NA

$t_statistic
[1] NA

$degrees_of_freedom
[1] NA

$p_value
[1] NA

$t_statistic
[1] NaN

$degrees_of_freedom
[1] NA

$p_value
[1] NA

$t_statistic
[1] NaN

$degrees_of_freedom
[1] NaN

$p_value
[1] NaN

$t_statistic
[1] -1.897367

$degrees_of_freedom
[1] 5.882353

$p_value
[1] 0.1075312

ERROR: Error in t_test(c("a", "b", "c"), c(1, 2, 3)): no loop for break/next, jumping to top level


## Exercise 4: Fibonacci Sequence

[Fibonacci sequence](https://en.wikipedia.org/wiki/Fibonacci_number) is
an integer sequence that frequently appears in mathematics and is often
associated with the [golden
ratio](https://en.wikipedia.org/wiki/Golden_ratio).

- Fibonacci sequence starts with the first element being equal to 0 and
  the second element being equal to 1.
- All subsequent elements are the sums of the preceeding two elements.
- More formally: $F_0 = 0, F_1 = 1, F_n = F_{n-1} + F_{n-2}, n > 1$
- The beginning of the sequence then looks like this: \[0, 1, 1, 2, 3,
  5, 8, …\]

Implement a function that takes a vector of integers as an input and
returns a vector of numbers from Fibonacci sequence, whose indices
correspond to the integers from the input vector.

Test the implemented function on the following inputs:

- `10`
- `c(1, 2, 3, 4, 5)`
- `c(3, 9, 12)`

Function specification:

Function takes 1 argument:

- `num_vec` - vector of whole numbers

Function returns 1 object:

- `fib` - vector of numbers from Fibonacci sequence corresponding to
  indices from the input vector

Example input → output:

- $[1, 3, 6]$ → $[0, 1, 5]$

In [15]:
# Exercise 4:

# Your code goes here
#The plan
	#1. Build the Fibonacci sequence far enough to reach the biggest position requested.
	#2. Pull out just the positions the user asked for.

#Step 1: make some variables for the function 

n <- 10                   # how many numbers we want, didn't work when i had it at zero originally 
fib_seq <- numeric(n)      # an empty vector with room for n numbers
fib_seq[1] <- 0            # first element is fixed at 0
fib_seq[2] <- 1            # second element is fixed at 1

for (i in 3:n) {           # start at position 3, since 1 and 2 are already filled
  fib_seq[i] <- fib_seq[i - 1] + fib_seq[i - 2]   # sum of the previous two
}

fib_numbers <- function(num_vec) {

  n <- max(num_vec)            # the largest position needed
  fib_seq <- numeric(n)        # empty vector with room for n numbers

  fib_seq[1] <- 0             # first element is fixed
  if (n >= 2) {
    fib_seq[2] <- 1           # second element is fixed as i couldn't work what to do elsewise 
  }


##ran into this issue so fixing it for future use:
#If n is 2, then 3:n is 3:2, and R counts downwards (3, 2), which would run the loop forever and mess the sequence. So we only run the loop when n is at least 3.
    
  if (n >= 3) {               # only loop when there's something to fill in
    for (i in 3:n) {
      fib_seq[i] <- fib_seq[i - 1] + fib_seq[i - 2]
    }
  }

    fib <- fib_seq[num_vec]
  return(fib)
}

#testing examples to see if code works 
fib_numbers(10)
fib_numbers(c(1, 2, 3, 4, 5))
fib_numbers(c(3, 9, 12))



[1] 34

[1] 0 1 1 2 3

[1]  1 21 89

## Exercise 5: Calculating t-tests

Assume that you drew a random sample of $200$ individuals and ran
$1,000$ experiments on them (an experiment does not need to mean
something big, think, different shapes of some button on a website or
Guinness served at slightly different temperatures).

Implement a function that takes a matrix of experimental results and a
vector with experimental/control-group assignment and returns a vector
of calculated t-statistics for each of the experiments. Internally, you
can use either the `t_test()` function from before or built-in
`t.test()` function in R. See simulated input data below.

Function specification:

Function takes 2 arguments:

- `mat` - matrix of experimental results (rows - individuals, columns -
  experiments)
- `grp` - vector of experimental/control-group assignment

Function returns 1 object:

- `tstats` - vector of calculated t-statistics for each of the
  experiments

set.seed(2026)
n <- 200
m <- 1000
mat <- matrix(rnorm(m * n, mean = 20, sd = 3), nrow = 200)
# For simplicity, let's assume that assignment to
# control and experimental groups is always the same.
grp <- rep(0:1, times = 100)

In [5]:
# Exercise 5:

# Your code goes 

set.seed(2026)
n <- 200
m <- 1000
mat <- matrix(rnorm(m * n, mean = 20, sd = 3), nrow = 200)
# For simplicity, let's assume that assignment to
# control and experimental groups is always the same.
grp <- rep(0:1, times = 100)

###heres my goals

#For each experiment (column), you:
	#1. Split that column's numbers into the control people and the experimental people.
	#2. Run a t-test comparing the two groups.
	#3. Keep the t-statistic.
#Then you return all 1000 t-statistics as one vector.
#with use of mat[, 1] pulls out column 1. The empty space before the comma means "all rows", so mat[, 5] is column 5, and so on.
# and col1[grp == 0] picks out the control people. 
#grp == 0 produces a vector of TRUE/FALSE values, one per person, and putting that inside square brackets keeps only the positions where it's TRUE
        ##similar to Q4

#Step 1: Don't get ahead of yourself- just do one, learned from mistake and then fixed errors

dim(mat)                       # check the shape of the matrix

col1 <- mat[, 1]               # all 200 people's results in experiment 1
control <- col1[grp == 0]      # results for the control group
treated <- col1[grp == 1]      # results for the experimental group

length(control)
length(treated)

t_test(control, treated)$t_statistic

###Both sizes are 100, and dim(mat) gives 200 rows (people) and 1000 columns (experiments). The t-statistic for experiment 1 is -1.69.

##Step 2: Repeat it for every column via a loop and get the t_test function from question 3. 


t_test <- function(x, y) {
  mean_x <- mean(x)
  mean_y <- mean(y)
  var_x <- var(x)
  var_y <- var(y)
  n_x <- length(x)
  n_y <- length(y)
  se_x <- sqrt(var_x/n_x)
  se_y <- sqrt(var_y/n_y)
  se <- sqrt(se_x^2 + se_y^2)
  t_stat <- (mean_x - mean_y) / se
  df <- se^4/(se_x^4/(n_x - 1) + se_y^4/(n_y - 1))
  p_val <- 2 * (1 - pt(abs(t_stat), df))
  res <- list(
    t_statistic = t_stat,
    degrees_of_freedom = df,
    p_value = p_val
  )
  return(res)
}


calc_tstats <- function(mat, grp) {

  m <- ncol(mat)
  tstats <- numeric(m)

  for (j in 1:m) {
    col <- mat[, j]
    control <- col[grp == 0]
    treated <- col[grp == 1]

    tstats[j] <- t_test(control, treated)$t_statistic
  }

  return(tstats)
}


tstats <- calc_tstats(mat, grp)

length(tstats)     # should be 1000
tstats[1]          # should be -1.69, matching what you got before
mean(abs(tstats) > 2)



##output was : 
    #1000
    #-1.69...
    #0.068


## the 0.068 means about 6.8% of the experiments had a t-statistic beyond ±2



[1]  200 1000

[1] 100

[1] 100

[1] -1.691847

[1] 1000

[1] -1.691847

[1] 0.068